In [7]:
# prompt: mount driver to group_project folder

import os

# Check if the group_project folder exists, create it if not
group_project_path = '/content/drive/MyDrive/group_project'
if not os.path.exists(group_project_path):
  os.makedirs(group_project_path)

# Now you can work with the group_project folder
print(f"group_project folder path: {group_project_path}")
!ls /content/drive/MyDrive/group_project


group_project folder path: /content/drive/MyDrive/group_project
combined_dev.tsv  combined_train.tsv  self_distillation.ipynb


In [9]:
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoModel
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
import json
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# # ------------------ Dataset ------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class CrisisClassifier(nn.Module):
    def __init__(self, input_size, output_size=11, hidden_dim=128, num_heads=4, dim_feedforward=2048,
                 num_layers=4, dropout=0.1, max_len=512,  device=device):
        super().__init__()
        self.embeddingL = nn.Embedding(input_size, hidden_dim)
        self.posembeddingL = nn.Embedding(max_len, hidden_dim)

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers)

        self.informative_linear = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

        self.gating_linear = nn.Sequential(
            nn.Linear(hidden_dim + 2, hidden_dim),
            nn.Sigmoid()
        )

        self.humanitarian_cat_linear = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, output_size)
        )

    def forward(self, x,attention_mask=None):
        pos_size = x.shape[1]
        pos_one_row = torch.arange(pos_size, device=x.device)
        pos = pos_one_row.expand(x.shape)

        embedding = self.embeddingL(x) + self.posembeddingL(pos)  # (batch_size, seq_len, hidden_dim)
        encoder_output = self.encoder(embedding)                 # (batch_size, seq_len, hidden_dim)
        pooled_output = encoder_output.mean(dim=1)               # (batch_size, hidden_dim)

        logits_informative = self.informative_linear(pooled_output)   # (batch_size, 2)
        probability_informative = F.softmax(logits_informative, dim=1)
        pooled_plus_probs = torch.cat([pooled_output, probability_informative], dim=1)

        logits_gating = self.gating_linear(pooled_plus_probs)     # (batch_size, hidden_dim)
        pooled_gating = pooled_output * logits_gating

        logits_humanitarian = self.humanitarian_cat_linear(pooled_gating)  # (batch_size, output_size)

        return logits_informative, logits_humanitarian

# ---------- Focal Loss ----------

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=0.0):
        super().__init__()
        assert gamma >= 0
        self.gamma = gamma
        self.weight = weight

    def forward(self, input, target):
        pt = F.softmax(input, dim=1)[range(input.shape[0]), target]
        log_pt = F.log_softmax(input, dim=1)[range(input.shape[0]), target]
        loss = self.weight[target] * (1 - pt).pow(self.gamma) * -log_pt
        return loss.mean()




In [10]:

class CrisisDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        encoding = self.tokenizer(
            row['text'],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'crisis_type': int(row["humanitarian_label"]),
            'informativeness': int(row["informativeness_label"]),
        }

# ------------------ Train Function ------------------
def train(model, dataloader, optimizer, criterion_informativeness, criterion_crisis, device, teacher_logits=None, alpha=0.8, alpha2=0.99):
    model.train()
    total_loss = 0

    for idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        crisis_labels = batch['crisis_type'].to(device)
        informativeness_labels = batch['informativeness'].to(device)

        optimizer.zero_grad()
        informativeness_logits, crisis_logits = model(input_ids, attention_mask)

        informativness_loss = criterion_informativeness(informativeness_logits, informativeness_labels)

        informative_mask = informativeness_labels == 1
        if informative_mask.sum() > 0:
            crisis_loss = criterion_crisis(crisis_logits[informative_mask], crisis_labels[informative_mask])
        else:
            loss2 = torch.tensor(0.0, device=device)

        loss = alpha2 * informativness_loss + (1-alpha2) * crisis_loss

        # 🔥 Self-distillation loss if teacher logits are provided
        if teacher_logits is not None:
            with torch.no_grad():
                t_info_logits, t_crisis_logits = teacher_logits[idx]
            distill_loss_info = F.kl_div(
                F.log_softmax(informativeness_logits / 1.0, dim=1),
                F.softmax(t_info_logits / 1.0, dim=1),
                reduction='batchmean'
            )
            distill_loss_crisis = F.kl_div(
                F.log_softmax(crisis_logits / 1.0, dim=1),
                F.softmax(t_crisis_logits / 1.0, dim=1),
                reduction='batchmean'
            )
            distill_loss = distill_loss_info + distill_loss_crisis
            loss = alpha * loss + (1 - alpha) * distill_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

# ------------------ Evaluation Function ------------------
def eval_model(model, dataloader, device):
    model.eval()
    all_info_preds, all_info_labels = [], []
    all_crisis_preds, all_crisis_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            info_true = batch["informativeness"].to(device)
            crisis_true = batch["crisis_type"].to(device)

            informativeness_logits, crisis_logits = model(input_ids, attention_mask)

            info_preds = torch.argmax(informativeness_logits, dim=1).cpu().numpy()
            crisis_preds = torch.argmax(crisis_logits, dim=1).cpu().numpy()

            all_info_preds.extend(info_preds)
            all_info_labels.extend(batch["informativeness"].cpu().numpy())
            all_crisis_preds.extend(crisis_preds)
            all_crisis_labels.extend(batch["crisis_type"].cpu().numpy())

    return np.array(all_info_labels), np.array(all_info_preds), np.array(all_crisis_labels), np.array(all_crisis_preds)

# ------------------ Plot Confusion Matrix ------------------
def plot_cm(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=300)
    plt.show()


In [11]:

# ------------------ Load Data ------------------
train_df = pd.read_csv("/content/drive/MyDrive/group_project/combined_train.tsv", sep="\t").dropna().reset_index(drop=True)
val_df = pd.read_csv("/content/drive/MyDrive/group_project/combined_dev.tsv", sep="\t").dropna().reset_index(drop=True)
info_labels = ['not_informative', 'informative']
info2id = {label: idx for idx, label in enumerate(info_labels)}
id2info = {idx: label for label, idx in info2id.items()}
hum_labels = sorted(train_df["humanitarian_label"].unique())
hum2id = {label: idx for idx, label in enumerate(hum_labels)}
id2hum = {idx: label for label, idx in hum2id.items()}

train_df["informativeness_label"] = train_df["informativeness_label"].map(info2id)
val_df["informativeness_label"] = val_df["informativeness_label"].map(info2id)
train_df["humanitarian_label"] = train_df["humanitarian_label"].map(hum2id)
val_df["humanitarian_label"] = val_df["humanitarian_label"].map(hum2id)

data_train = CrisisDataset(train_df, tokenizer, max_len=128)
data_val = CrisisDataset(val_df, tokenizer, max_len=128)

train_loader = DataLoader(data_train, batch_size=32, shuffle=True)
val_loader = DataLoader(data_val, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


In [12]:
# ------------------ Step 1: Train Teacher Model ------------------
model_teacher = CrisisClassifier(input_size=tokenizer.vocab_size).to(device)
optimizer = torch.optim.AdamW(model_teacher.parameters(), lr=2e-5)
criterion_informativeness = nn.CrossEntropyLoss()
criterion_crisis = FocalLoss(weight=torch.ones(len(hum_labels), device=device), gamma=2.0) # Assuming hum_labels contains all possible crisis labels
epochs = 10
print("\n🛠️ Training Teacher Model...")
for epoch in range(epochs):
    train_loss = train(model_teacher, train_loader, optimizer, criterion_informativeness, criterion_crisis, device)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

# Evaluate teacher
info_true, info_preds, crisis_true, crisis_preds = eval_model(model_teacher, val_loader, device)

print("\n📜 Classification Report (Teacher)")
# Now print the report
print(classification_report(info_true, info_preds, target_names=[id2info[i] for i in sorted(id2info)]))
# Find all unique labels in crisis_true and crisis_preds
present_labels = sorted(set(np.unique(crisis_true)).union(np.unique(crisis_preds)))
print(classification_report( crisis_true, crisis_preds, labels=present_labels, target_names=[id2hum[i] for i in present_labels], zero_division=0 ))

# Plot the Confusion Matrix
plot_cm(info_true, info_preds, [id2info[i] for i in sorted(id2info)], "Teacher_Informativeness")
plot_cm(crisis_true, crisis_preds, [id2hum[i] for i in sorted(id2hum)], "Teacher_Crisis")

# Save teacher logits
teacher_logits = []
with torch.no_grad():
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        info_logits, crisis_logits = model_teacher(input_ids, attention_mask)
        teacher_logits.append((info_logits.detach(), crisis_logits.detach()))



🛠️ Training Teacher Model...


KeyboardInterrupt: 

In [ ]:
# ------------------ Step 2: Train Student Model with Self-Distillation ------------------
model_student = CrisisClassifier(input_size=tokenizer.vocab_size).to(device)
optimizer = torch.optim.AdamW(model_student.parameters(), lr=2e-5)

print("\n🛠️ Training Student Model (Self-Distillation)...")
for epoch in range(epochs):
    train_loss = train(model_student, train_loader, optimizer, criterion_informativeness, criterion_crisis, device, teacher_logits, alpha=0.5)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

# Evaluate student
info_true, info_preds, crisis_true, crisis_preds = eval_model(model_student, val_loader, device)

print("\n📜 Classification Report (Student)")
print(classification_report(info_true, info_preds, target_names=[id2info[i] for i in sorted(id2info)]))

# Find all unique labels in crisis_true and crisis_preds
present_labels = sorted(set(np.unique(crisis_true)).union(np.unique(crisis_preds)))
print(classification_report( crisis_true, crisis_preds, labels=present_labels, target_names=[id2hum[i] for i in present_labels], zero_division=0 ))

plot_cm(info_true, info_preds, [id2info[i] for i in sorted(id2info)], "Student_Informativeness")
plot_cm(crisis_true, crisis_preds, [id2hum[i] for i in sorted(id2hum)], "Student_Crisis")
